# 13 - Build analytics dimensions

Builds physical Delta lookup tables for Date, Time, Camera, Location, Video, and ModelConfig. Small dimensions are rebuilt from authoritative successful work; the video dimension is merged incrementally unless a full rebuild is requested.

**Before the first run:** Run `00_bootstrap_lakehouse.ipynb`. New fact partitions receive their relationship keys from `07_build_gold_aggregates.ipynb`; this notebook backfills missing keys on fact rows created before the dimension schema was introduced.

**After importing into Fabric:** On the configuration code cell, select **... -> Toggle parameter cell** and confirm the parameter indicator. Then attach and pin `people_counter_<environment>` as this notebook's default Lakehouse.

In [ ]:
LOOKBACK_HOURS = 48
FULL_REBUILD = False
DATABASE = ""
TABLE_PREFIX = "people_counter"

In [ ]:
from datetime import datetime, timedelta, timezone
import json
import re

import notebookutils
from delta.tables import DeltaTable
from pyspark.sql import DataFrame, SparkSession, functions as F


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def parse_bool(value: object, name: str) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, str) and value.strip().lower() in {"true", "false"}:
        return value.strip().lower() == "true"
    raise ValueError(f"{name} must be true or false")


database = DATABASE.strip()
prefix = TABLE_PREFIX.strip()
if database and IDENTIFIER.fullmatch(database) is None:
    raise ValueError("DATABASE is not a valid identifier")
if IDENTIFIER.fullmatch(prefix) is None:
    raise ValueError("TABLE_PREFIX is not a valid identifier")
if isinstance(LOOKBACK_HOURS, bool):
    raise ValueError("LOOKBACK_HOURS must be an integer")
lookback_hours = int(LOOKBACK_HOURS)
if lookback_hours < 1:
    raise ValueError("LOOKBACK_HOURS must be at least 1")
full_rebuild = parse_bool(FULL_REBUILD, "FULL_REBUILD")


def table(suffix: str) -> str:
    value = f"{prefix}_{suffix}"
    return f"{database}.{value}" if database else value


def replace_dimension(target_table: str, source: DataFrame) -> int:
    rows = source.count()
    source.write.format("delta").mode("overwrite").saveAsTable(target_table)
    return rows


spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("A Fabric Spark session is required")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")
now = datetime.now(timezone.utc)
cutoff = now - timedelta(hours=lookback_hours)

work = spark_session.table(table("video_work"))
successful_work = work.where(
    (F.col("status") == "SUCCEEDED") & F.col("committed_attempt_id").isNotNull()
)

duplicate_work = successful_work.groupBy("work_id").count().where(F.col("count") != 1).limit(10).collect()
if duplicate_work:
    raise RuntimeError(f"Duplicate successful work_id values prevent DimVideo creation: {duplicate_work}")

def backfill_time_key(suffix: str, timestamp_column: str) -> None:
    target_table = table(suffix)
    missing_key = spark_session.table(target_table).where(
        F.col("time_key").isNull() & F.col(timestamp_column).isNotNull()
    )
    if missing_key.head(1):
        DeltaTable.forName(spark_session, target_table).update(
            condition=f"time_key IS NULL AND {timestamp_column} IS NOT NULL",
            set={"time_key": f"hour({timestamp_column}) * 60 + minute({timestamp_column})"},
        )


backfill_time_key("gold_flow_minute", "minute_utc")
backfill_time_key("gold_flow_hour", "hour_utc")
backfill_time_key("gold_operations_hour", "hour_utc")

video_relationship_keys = successful_work.select(
    "work_id",
    "config_sha256",
    (F.hour("captured_at_utc") * 60 + F.minute("captured_at_utc")).cast("int").alias("time_key"),
)
(
    DeltaTable.forName(spark_session, table("gold_video"))
    .alias("t")
    .merge(video_relationship_keys.alias("s"), "t.work_id = s.work_id")
    .whenMatchedUpdate(
        condition="t.time_key IS NULL OR t.config_sha256 IS NULL",
        set={"time_key": "s.time_key", "config_sha256": "s.config_sha256"},
    )
    .execute()
)

required_fact_keys = {
    "gold_flow_minute": ("time_key",),
    "gold_flow_hour": ("time_key",),
    "gold_video": ("time_key", "config_sha256"),
    "gold_operations_hour": ("time_key",),
}
missing_fact_keys = []
for suffix, columns in required_fact_keys.items():
    missing_condition = F.lit(False)
    for column in columns:
        missing_condition = missing_condition | F.col(column).isNull()
    if spark_session.table(table(suffix)).where(missing_condition).head(1):
        missing_fact_keys.append(f"{suffix}: {', '.join(columns)}")
if missing_fact_keys:
    raise RuntimeError(
        "Gold fact relationship keys are incomplete after migration: "
        + "; ".join(missing_fact_keys)
    )

camera_conflicts = (
    successful_work.groupBy("camera_id")
    .agg(
        F.countDistinct("location_id").alias("location_count"),
        F.countDistinct("camera_timezone").alias("timezone_count"),
    )
    .where((F.col("location_count") != 1) | (F.col("timezone_count") != 1))
    .limit(10)
    .collect()
)
if camera_conflicts:
    raise RuntimeError(
        "Each camera_id must map to exactly one location_id and camera_timezone: "
        f"{camera_conflicts}"
    )

config_conflicts = (
    successful_work.groupBy("config_sha256")
    .agg(F.countDistinct("config_json").alias("config_json_count"))
    .where(F.col("config_json_count") != 1)
    .limit(10)
    .collect()
)
if config_conflicts:
    raise RuntimeError(
        "Each config_sha256 must map to exactly one config_json value: "
        f"{config_conflicts}"
    )

date_values = (
    spark_session.table(table("gold_flow_minute")).select(F.col("flow_date").alias("date_key"))
    .unionByName(spark_session.table(table("gold_flow_hour")).select(F.col("flow_date").alias("date_key")))
    .unionByName(spark_session.table(table("gold_video")).select(F.col("capture_date").alias("date_key")))
    .unionByName(
        spark_session.table(table("gold_operations_hour")).select(
            F.col("operation_date").alias("date_key")
        )
    )
    .where(F.col("date_key").isNotNull())
)
date_bounds = date_values.agg(F.min("date_key").alias("start_date"), F.max("date_key").alias("end_date")).first()
start_date = date_bounds.start_date or now.date()
end_date = date_bounds.end_date or now.date()
date_count = (end_date - start_date).days + 1
dim_date = (
    spark_session.range(date_count)
    .select(F.date_add(F.lit(start_date), F.col("id").cast("int")).alias("date_key"))
    .withColumn("calendar_year", F.year("date_key").cast("int"))
    .withColumn("calendar_quarter", F.quarter("date_key").cast("int"))
    .withColumn("calendar_month", F.month("date_key").cast("int"))
    .withColumn("month_name", F.date_format("date_key", "MMMM"))
    .withColumn("month_short_name", F.date_format("date_key", "MMM"))
    .withColumn("year_month", F.date_format("date_key", "yyyy-MM"))
    .withColumn("day_of_month", F.dayofmonth("date_key").cast("int"))
    .withColumn("iso_day_of_week", (((F.dayofweek("date_key") + 5) % 7) + 1).cast("int"))
    .withColumn("iso_week_year", F.year(F.date_add("date_key", 4 - F.col("iso_day_of_week"))).cast("int"))
    .withColumn("iso_week_of_year", F.weekofyear("date_key").cast("int"))
    .withColumn(
        "iso_year_week",
        F.concat(
            F.col("iso_week_year").cast("string"),
            F.lit("-W"),
            F.lpad(F.col("iso_week_of_year").cast("string"), 2, "0"),
        ),
    )
    .withColumn("day_name", F.date_format("date_key", "EEEE"))
    .withColumn("is_weekend", F.col("iso_day_of_week").isin(6, 7))
    .withColumn("refreshed_at", F.lit(now))
)

dim_time = (
    spark_session.range(1440)
    .select(F.col("id").cast("int").alias("time_key"))
    .withColumn("hour_24", F.floor(F.col("time_key") / 60).cast("int"))
    .withColumn("minute_of_hour", (F.col("time_key") % 60).cast("int"))
    .withColumn("time_label", F.format_string("%02d:%02d", F.col("hour_24"), F.col("minute_of_hour")))
    .withColumn("hour_label", F.format_string("%02d:00", F.col("hour_24")))
    .withColumn(
        "day_part",
        F.when(F.col("hour_24") < 6, "Night")
        .when(F.col("hour_24") < 12, "Morning")
        .when(F.col("hour_24") < 18, "Afternoon")
        .otherwise("Evening"),
    )
    .withColumn("refreshed_at", F.lit(now))
)

dim_camera = (
    successful_work.groupBy("camera_id")
    .agg(
        F.first("location_id").alias("location_id"),
        F.first("camera_timezone").alias("camera_timezone"),
        F.min("captured_at_utc").alias("first_capture_utc"),
        F.max("captured_at_utc").alias("last_capture_utc"),
        F.countDistinct("work_id").cast("long").alias("video_count"),
    )
    .withColumn("refreshed_at", F.lit(now))
)
dim_location = (
    successful_work.groupBy("location_id")
    .agg(
        F.min("captured_at_utc").alias("first_capture_utc"),
        F.max("captured_at_utc").alias("last_capture_utc"),
        F.countDistinct("camera_id").cast("long").alias("camera_count"),
        F.countDistinct("work_id").cast("long").alias("video_count"),
    )
    .withColumn("refreshed_at", F.lit(now))
)

config_base = successful_work.groupBy("config_sha256", "config_json").agg(
    F.min("captured_at_utc").alias("first_capture_utc"),
    F.max("captured_at_utc").alias("last_capture_utc"),
    F.countDistinct("work_id").cast("long").alias("video_count"),
)
dim_model_config = config_base.select(
    "config_sha256",
    "config_json",
    F.get_json_object("config_json", "$.pipeline").alias("pipeline"),
    F.get_json_object("config_json", "$.device_variant").alias("device_variant"),
    F.get_json_object("config_json", "$.device").alias("device"),
    F.get_json_object("config_json", "$.batch_size").cast("int").alias("batch_size"),
    F.get_json_object("config_json", "$.sample_fps").cast("double").alias("sample_fps"),
    F.get_json_object("config_json", "$.detection_threshold").cast("double").alias("detection_threshold"),
    F.get_json_object("config_json", "$.use_fp16").cast("boolean").alias("use_fp16"),
    F.get_json_object("config_json", "$.detector_model").alias("detector_model"),
    F.get_json_object("config_json", "$.camera_motion_compensation").cast("boolean").alias("camera_motion_compensation"),
    F.get_json_object("config_json", "$.line").alias("counting_line_json"),
    "first_capture_utc",
    "last_capture_utc",
    "video_count",
    F.lit(now).alias("refreshed_at"),
)

video_target = table("gold_dim_video")
video_target_is_empty = not spark_session.table(video_target).head(1)
video_source = successful_work
if not full_rebuild and not video_target_is_empty:
    video_source = video_source.where(F.col("completed_at") >= F.lit(cutoff))
dim_video = video_source.select(
    "work_id",
    "asset_id",
    "asset_version",
    "camera_id",
    "location_id",
    "captured_at_utc",
    "capture_date",
    (F.hour("captured_at_utc") * 60 + F.minute("captured_at_utc")).cast("int").alias("time_key"),
    "camera_timezone",
    "config_sha256",
    F.lit(now).alias("refreshed_at"),
)
video_rows_processed = dim_video.count()
if full_rebuild or video_target_is_empty:
    (
        dim_video.write.format("delta")
        .mode("overwrite")
        .partitionBy("capture_date")
        .saveAsTable(video_target)
    )
else:
    (
        DeltaTable.forName(spark_session, video_target)
        .alias("t")
        .merge(dim_video.alias("s"), "t.work_id = s.work_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

dimension_rows = {
    "date_rows": replace_dimension(table("gold_dim_date"), dim_date),
    "time_rows": replace_dimension(table("gold_dim_time"), dim_time),
    "camera_rows": replace_dimension(table("gold_dim_camera"), dim_camera),
    "location_rows": replace_dimension(table("gold_dim_location"), dim_location),
    "model_config_rows": replace_dimension(table("gold_dim_model_config"), dim_model_config),
}

relationships = (
    ("gold_flow_minute", "flow_date", "gold_dim_date", "date_key"),
    ("gold_flow_hour", "flow_date", "gold_dim_date", "date_key"),
    ("gold_video", "capture_date", "gold_dim_date", "date_key"),
    ("gold_operations_hour", "operation_date", "gold_dim_date", "date_key"),
    ("gold_flow_minute", "time_key", "gold_dim_time", "time_key"),
    ("gold_flow_hour", "time_key", "gold_dim_time", "time_key"),
    ("gold_video", "time_key", "gold_dim_time", "time_key"),
    ("gold_operations_hour", "time_key", "gold_dim_time", "time_key"),
    ("gold_flow_minute", "camera_id", "gold_dim_camera", "camera_id"),
    ("gold_flow_hour", "camera_id", "gold_dim_camera", "camera_id"),
    ("gold_video", "camera_id", "gold_dim_camera", "camera_id"),
    ("gold_flow_minute", "location_id", "gold_dim_location", "location_id"),
    ("gold_flow_hour", "location_id", "gold_dim_location", "location_id"),
    ("gold_video", "location_id", "gold_dim_location", "location_id"),
    ("gold_video", "config_sha256", "gold_dim_model_config", "config_sha256"),
    ("gold_video", "work_id", "gold_dim_video", "work_id"),
)
unresolved_relationships = []
for fact_suffix, fact_key, dimension_suffix, dimension_key in relationships:
    fact_keys = (
        spark_session.table(table(fact_suffix))
        .select(F.col(fact_key).alias("relationship_key"))
        .where(F.col("relationship_key").isNotNull())
        .distinct()
    )
    dimension_keys = spark_session.table(table(dimension_suffix)).select(
        F.col(dimension_key).alias("relationship_key")
    )
    if fact_keys.join(dimension_keys, "relationship_key", "left_anti").head(1):
        unresolved_relationships.append(
            f"{fact_suffix}[{fact_key}] -> {dimension_suffix}[{dimension_key}]"
        )
if unresolved_relationships:
    raise RuntimeError(
        "Gold fact keys have no matching dimension row: "
        + "; ".join(unresolved_relationships)
    )

outcome = {
    "refreshed_at": now.isoformat(),
    "lookback_hours": lookback_hours,
    "full_rebuild": full_rebuild or video_target_is_empty,
    **dimension_rows,
    "video_rows_processed": video_rows_processed,
    "video_rows_total": spark_session.table(video_target).count(),
    "relationships_validated": len(relationships),
}
print(json.dumps(outcome, sort_keys=True))

In [ ]:
notebookutils.notebook.exit(json.dumps(outcome, sort_keys=True))